# LUAD Survival Analysis — DeepSurv (Leakage-Free)
**Author:** Parth Shringarpure  
**Date:** June 2026  

## What makes this notebook different from 04_deepsurv.ipynb
The previous notebook selected 1,000 genes on the full 478-patient dataset BEFORE 
cross-validation. This means test fold patients helped choose their own features — 
classic label leakage through feature selection.

**This notebook fixes that by:**
1. Loading the full 20,502-gene raw matrix
2. Performing ALL preprocessing (variance filter + univariate Cox + scaling) 
   INSIDE each CV fold, using only that fold's training patients
3. The test fold never touches any preprocessing step

This produces a genuinely unbiased C-index estimate.

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

os.chdir('/Users/parthshringarpure/Desktop/AI/Projects/luad_survival')

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from lifelines.utils import concordance_index
from lifelines import CoxPHFitter

torch.manual_seed(42)
np.random.seed(42)

print("All imports successful")
print(f"Working directory: {os.getcwd()}")

All imports successful
Working directory: /Users/parthshringarpure/Desktop/AI/Projects/luad_survival


## Step 2: Load Clinical Survival Labels and Full Raw Gene Matrix

We load the FULL 20,502-gene matrix here — NOT the pre-selected expression_matrix.csv.
Gene selection happens inside each CV fold below.

In [8]:
clinical = pd.read_csv('data/processed/clinical_survival.csv', index_col=0)
print(f"Clinical patients: {clinical.shape[0]}")

# Load FULL raw RNA-seq matrix
RNASEQ_PATH = (
    "data/raw/gdac.broadinstitute.org_LUAD.Merge_rnaseqv2__illuminahiseq_rnaseqv2"
    "__unc_edu__Level_3__RSEM_genes_normalized__data.Level_3.2016012800.0.0/"
    "LUAD.rnaseqv2__illuminahiseq_rnaseqv2__unc_edu__Level_3__RSEM_genes_normalized"
    "__data.data.txt"
)

print("Loading full RNA-seq matrix — takes ~20 seconds...")
rna_raw = pd.read_csv(
    RNASEQ_PATH, sep='\t',
    index_col=0, skiprows=[1],
    low_memory=False
)

# Same cleaning as notebook 02
rna_named = rna_raw[~rna_raw.index.str.startswith('?')]
rna_named.index = rna_named.index.str.split('|').str[0]
rna_named.columns = rna_named.columns.str[:12].str.lower()
rna = rna_named.T
rna = rna[~rna.index.duplicated(keep='first')]

# Match to clinical
common = clinical.index.intersection(rna.index)
rna = rna.loc[common]
clinical = clinical.loc[common]


# log2(x+1) normalise
rna_log = np.log2(rna.astype(float) + 1)

print(f"Full gene matrix: {rna_log.shape}  (patients × genes)")
print(f"Clinical matched: {clinical.shape[0]} patients")

Clinical patients: 484
Loading full RNA-seq matrix — takes ~20 seconds...
Full gene matrix: (478, 20502)  (patients × genes)
Clinical matched: 478 patients


## Step 3: Define Model, Loss Function and Dataset Classes

In [9]:
class DeepSurv(nn.Module):
    def __init__(self, input_dim, hidden_dims, dropout):
        super(DeepSurv, self).__init__()
        layers = []
        prev_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.BatchNorm1d(hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev_dim = hidden_dim
        layers.append(nn.Linear(prev_dim, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x).squeeze()


def cox_partial_likelihood_loss(risk_scores, survival_times, events):
    order = torch.argsort(survival_times, descending=True)
    risk_scores = risk_scores[order]
    events      = events[order]
    risk_scores = risk_scores - risk_scores.max()
    log_cumsum  = torch.log(torch.cumsum(torch.exp(risk_scores), dim=0))
    loss        = -torch.mean((risk_scores - log_cumsum) * events)
    return loss


class SurvivalDataset(Dataset):
    def __init__(self, X, y_time, y_event):
        self.X       = torch.FloatTensor(X)
        self.y_time  = torch.FloatTensor(y_time)
        self.y_event = torch.FloatTensor(y_event)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y_time[idx], self.y_event[idx]


print("Model classes defined")
print(f"Architecture: input → 64 → 32 → 1")
print(f"Dropout: 0.5, Weight decay: 1e-3")

Model classes defined
Architecture: input → 64 → 32 → 1
Dropout: 0.5, Weight decay: 1e-3


## Step 4: Gene Selection Function

This function runs inside each CV fold on training patients only.
It performs:
1. Variance filter → top 5,000 genes
2. Univariate Cox → top 1,000 genes by survival p-value

The test fold patients are NEVER passed to this function.

In [10]:
def select_genes_on_train(X_train_df, y_time_train, y_event_train,
                           n_variance=5000, n_cox=1000):
    """
    Select top genes using ONLY training patients.
    
    Args:
        X_train_df:    DataFrame, shape (n_train, 20502)
        y_time_train:  array of survival times
        y_event_train: array of event indicators
        n_variance:    genes to keep after variance filter
        n_cox:         genes to keep after Cox filter
    
    Returns:
        selected_genes: list of gene names to use
    """
    # Stage 1 — Variance filter
    gene_var = X_train_df.var(axis=0)
    top_var_genes = gene_var.nlargest(n_variance).index
    X_var = X_train_df[top_var_genes]

    # Stage 2 — Univariate Cox on training patients only
    cox_pvals = {}
    for gene in X_var.columns:
        df = pd.DataFrame({
            'survival_time': y_time_train,
            'event':         y_event_train,
            'gene':          X_var[gene].values
        }).dropna()
        try:
            cph = CoxPHFitter()
            cph.fit(df, duration_col='survival_time', event_col='event')
            cox_pvals[gene] = cph.summary.loc['gene', 'p']
        except Exception:
            cox_pvals[gene] = 1.0

    # Keep top n_cox genes by p-value
    pval_series = pd.Series(cox_pvals).sort_values()
    selected_genes = pval_series.head(n_cox).index.tolist()

    return selected_genes


print("Gene selection function defined")
print("This runs inside each fold — test patients never seen during selection")

Gene selection function defined
This runs inside each fold — test patients never seen during selection


## Step 5: Leakage-Free 5-Fold Cross-Validation

For each fold:
1. Split into train (80%) and test (20%)
2. Select 1,000 genes using ONLY train patients
3. Scale using ONLY train patients
4. Train DeepSurv on train patients
5. Evaluate on test patients — completely unseen until this moment
6. Record C-index for this fold

Average across 5 folds = honest C-index estimate

In [11]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Convert to numpy for indexing
X_raw       = rna_log.values.astype(np.float32)
gene_names  = rna_log.columns.tolist()
y_time_all  = clinical['survival_time'].values.astype(np.float32)
y_event_all = clinical['event'].values.astype(np.float32)

fold_cindices = []

print("Running leakage-free 5-fold cross-validation...")
print("Each fold runs gene selection on training patients only")
print(f"{'Fold':>6} {'Genes selected':>16} {'Best Epoch':>12} {'Test C-index':>14}")
print("-" * 55)

for fold, (train_idx, test_idx) in enumerate(kf.split(X_raw)):

    # ── 1. Split ──────────────────────────────────────────
    X_f_train_raw = rna_log.iloc[train_idx]   # DataFrame — needed for gene selection
    X_f_test_raw  = rna_log.iloc[test_idx]

    t_train = y_time_all[train_idx]
    e_train = y_event_all[train_idx]
    t_test  = y_time_all[test_idx]
    e_test  = y_event_all[test_idx]

    # ── 2. Gene selection on train only ───────────────────
    selected_genes = select_genes_on_train(
        X_f_train_raw, t_train, e_train,
        n_variance=5000, n_cox=1000
    )

    # Apply same gene selection to test
    X_f_train = X_f_train_raw[selected_genes].values.astype(np.float32)
    X_f_test  = X_f_test_raw[selected_genes].values.astype(np.float32)

    # ── 3. Scale on train only ────────────────────────────
    fold_scaler = StandardScaler()
    X_f_train   = fold_scaler.fit_transform(X_f_train)
    X_f_test    = fold_scaler.transform(X_f_test)

    # ── 4. Train/val split within training fold ───────────
    n_val = int(0.2 * len(X_f_train))
    X_tr, X_val = X_f_train[:-n_val], X_f_train[-n_val:]
    t_tr, t_val = t_train[:-n_val],   t_train[-n_val:]
    e_tr, e_val = e_train[:-n_val],   e_train[-n_val:]

    # ── 5. Build model ────────────────────────────────────
    fold_model = DeepSurv(
        input_dim   = 1000,
        hidden_dims = [64, 32],
        dropout     = 0.5
    )
    fold_optimizer = torch.optim.Adam(
        fold_model.parameters(),
        lr=0.001,
        weight_decay=1e-3
    )
    fold_loader = DataLoader(
        SurvivalDataset(X_tr, t_tr, e_tr),
        batch_size=64, shuffle=True
    )

    # ── 6. Train ──────────────────────────────────────────
    best_cindex  = 0.0
    best_weights = None
    best_epoch   = 0
    no_improve   = 0

    for epoch in range(1, 200):
        fold_model.train()
        for X_batch, t_batch, e_batch in fold_loader:
            fold_optimizer.zero_grad()
            risk = fold_model(X_batch)
            loss = cox_partial_likelihood_loss(risk, t_batch, e_batch)
            loss.backward()
            fold_optimizer.step()

        fold_model.eval()
        with torch.no_grad():
            val_risk = fold_model(torch.FloatTensor(X_val)).numpy()
        cindex = concordance_index(t_val, -val_risk, e_val)

        if cindex > best_cindex:
            best_cindex  = cindex
            best_weights = {k: v.clone() for k, v in fold_model.state_dict().items()}
            best_epoch   = epoch
            no_improve   = 0
        else:
            no_improve += 1

        if no_improve >= 20:
            break

    # ── 7. Evaluate on test fold ──────────────────────────
    fold_model.load_state_dict(best_weights)
    fold_model.eval()
    with torch.no_grad():
        test_risk = fold_model(torch.FloatTensor(X_f_test)).numpy()
    test_cindex = concordance_index(t_test, -test_risk, e_test)

    fold_cindices.append(test_cindex)
    print(f"{fold+1:>6} {len(selected_genes):>16} {best_epoch:>12} {test_cindex:>14.4f}")

mean_cindex = np.mean(fold_cindices)
std_cindex  = np.std(fold_cindices)

print(f"\n{'='*55}")
print(f"DEEPSURV LEAKAGE-FREE 5-FOLD CV RESULTS")
print(f"{'='*55}")
print(f"Fold C-indices: {[f'{c:.4f}' for c in fold_cindices]}")
print(f"Mean C-index:   {mean_cindex:.4f} ± {std_cindex:.4f}")
print(f"{'='*55}")
print(f"\nHONEST MODEL COMPARISON:")
print(f"  Cox clinical:                0.700")
print(f"  Cox-Lasso expression:        0.649  (note: also slightly leaky)")
print(f"  DeepSurv (old, leaky):       0.712")
print(f"  DeepSurv (leakage-free):     {mean_cindex:.3f} ± {std_cindex:.3f}")

Running leakage-free 5-fold cross-validation...
Each fold runs gene selection on training patients only
  Fold   Genes selected   Best Epoch   Test C-index
-------------------------------------------------------
     1             1000           29         0.5085
     2             1000           29         0.6687
     3             1000           35         0.5168
     4             1000            1         0.5834
     5             1000            4         0.5693

DEEPSURV LEAKAGE-FREE 5-FOLD CV RESULTS
Fold C-indices: ['0.5085', '0.6687', '0.5168', '0.5834', '0.5693']
Mean C-index:   0.5693 ± 0.0575

HONEST MODEL COMPARISON:
  Cox clinical:                0.700
  Cox-Lasso expression:        0.649  (note: also slightly leaky)
  DeepSurv (old, leaky):       0.712
  DeepSurv (leakage-free):     0.569 ± 0.058


In [12]:
import pickle
import json
import torch

# Retrain on full data for deployment
print("Retraining DeepSurv on full dataset...")

X_full_raw = rna_log.values.astype(np.float32)
y_time_full  = clinical['survival_time'].values.astype(np.float32)
y_event_full = clinical['event'].values.astype(np.float32)

# Gene selection on full data for deployment model
# (deployment model trained on all data — no CV needed)
print("Selecting genes on full dataset...")
selected_genes_full = select_genes_on_train(
    rna_log, y_time_full, y_event_full,
    n_variance=5000, n_cox=1000
)
X_full_selected = rna_log[selected_genes_full].values.astype(np.float32)

# Scale
full_scaler = StandardScaler()
X_full_scaled = full_scaler.fit_transform(X_full_selected)

# Build and train model
final_model = DeepSurv(input_dim=1000, hidden_dims=[64, 32], dropout=0.5)
final_optimizer = torch.optim.Adam(
    final_model.parameters(), lr=0.001, weight_decay=1e-3
)
full_loader = DataLoader(
    SurvivalDataset(X_full_scaled, y_time_full, y_event_full),
    batch_size=64, shuffle=True
)

best_loss = float('inf')
best_weights = None

for epoch in range(1, 40):
    final_model.train()
    epoch_loss = 0.0
    for X_batch, t_batch, e_batch in full_loader:
        final_optimizer.zero_grad()
        risk = final_model(X_batch)
        loss = cox_partial_likelihood_loss(risk, t_batch, e_batch)
        loss.backward()
        final_optimizer.step()
        epoch_loss += loss.item()
    avg_loss = epoch_loss / len(full_loader)
    if avg_loss < best_loss:
        best_loss = avg_loss
        best_weights = {k: v.clone() for k, v in final_model.state_dict().items()}

final_model.load_state_dict(best_weights)
print(f"Training complete. Best loss: {best_loss:.4f}")

# Save everything
torch.save(final_model.state_dict(), 'models/deepsurv_final.pt')
print("Saved: models/deepsurv_final.pt")

with open('models/scaler_deepsurv.pkl', 'wb') as f:
    pickle.dump(full_scaler, f)
print("Saved: models/scaler_deepsurv.pkl")

config = {
    'input_dim'   : 1000,
    'hidden_dims' : [64, 32],
    'dropout'     : 0.5,
    'cv_cindex'   : 0.569,
    'cv_std'      : 0.058,
    'n_patients'  : 478,
    'n_genes'     : 1000
}
with open('models/deepsurv_config.json', 'w') as f:
    json.dump(config, f, indent=2)
print("Saved: models/deepsurv_config.json")

# Save deployment gene list
with open('models/gene_list.json', 'w') as f:
    json.dump(selected_genes_full, f)
print(f"Saved: models/gene_list.json ({len(selected_genes_full)} genes)")

# Final check
import os
print(f"\nAll saved models:")
for f in sorted(os.listdir('models/')):
    size = os.path.getsize(f'models/{f}')
    print(f"  {f:45s} {size/1024:.1f} KB")

Retraining DeepSurv on full dataset...
Selecting genes on full dataset...
Training complete. Best loss: 0.4092
Saved: models/deepsurv_final.pt
Saved: models/scaler_deepsurv.pkl
Saved: models/deepsurv_config.json
Saved: models/gene_list.json (1000 genes)

All saved models:
  .DS_Store                                     6.0 KB
  cox_clinical.pkl                              49.8 KB
  cox_lasso_expression.pkl                      23.0 KB
  cox_lasso_summary.json                        0.4 KB
  deepsurv_config.json                          0.2 KB
  deepsurv_final.pt                             265.9 KB
  gene_list.json                                9.5 KB
  scaler_deepsurv.pkl                           23.9 KB
  scaler_expression.pkl                         32.5 KB
